In [30]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

# Actividades

Mendoza Hernández Carlos Emiliano

1. Basándose en la información contenida en el archivo `plane_routes.csv`, realice una infografía de las operaciones realizadas por los aeropuertos del mundo así como de las rutas aéreas seguidas por las aeronaves. La infografía deberá mostrar:

    a. Las operaciones realizadas por cada aeropuerto, georeferenciado.

    b. El trazo de las rutas seguidas por las aeronaves

    c. Las 20 aeronaves más usadas a nivel global

### Preproceso de los datos

In [31]:
df1 = pd.read_csv('./data/plane_routes.csv', usecols=['source airport', 'destination airport', 'equipment'])
df1.dropna(subset=['source airport', 'destination airport'], inplace=True)
df1

,source airport,destination airport,equipment
0,AER,KZN,CR2
1,ASF,KZN,CR2
2,ASF,MRV,CR2
3,CEK,KZN,CR2
4,CEK,OVB,CR2
...,...,...,...
67658,WYA,ADL,SF3
67659,DME,FRU,734
67660,FRU,DME,734
67661,FRU,OSS,734


In [32]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67663 entries, 0 to 67662
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   source airport       67663 non-null  object
 1   destination airport  67663 non-null  object
 2   equipment            67645 non-null  object
dtypes: object(3)
memory usage: 1.5+ MB


In [33]:
df2 = pd.read_excel('./data/GlobalAirportDatabase.xlsx', usecols=['IATA Code', 'Name', 'City/Town', 'Country', 'Latitude Decimal Degrees', 'Longitude Decimal Degrees'])
df2.dropna(subset=['IATA Code'], inplace=True)
df2.dropna(subset=['Latitude Decimal Degrees', 'Longitude Decimal Degrees'], inplace=True)
df2 = df2[(df2['Latitude Decimal Degrees'] != 0.0) & (df2['Longitude Decimal Degrees'] != 0.0)]
df2

,IATA Code,Name,City/Town,Country,Latitude Decimal Degrees,Longitude Decimal Degrees
0,GKA,GOROKA,GOROKA,PAPUA NEW GUINEA,-6.082,145.392
2,MAG,MADANG,MADANG,PAPUA NEW GUINEA,-5.207,145.789
3,HGU,MOUNT HAGEN,MOUNT HAGEN,PAPUA NEW GUINEA,-5.826,144.296
4,LAE,NADZAB,NADZAB,PAPUA NEW GUINEA,-6.570,146.726
5,POM,PORT MORESBY JACKSONS INTERNATIONAL,PORT MORESBY,PAPUA NEW GUINEA,-9.443,147.220
...,...,...,...,...,...,...
9275,KHG,KASHI,KASHI,CHINA,39.543,76.022
9276,HTN,HOTAN,HOTAN,CHINA,37.038,79.866
9278,URC,DIWOPU,URUMQI,CHINA,43.907,87.474
9286,HRB,TAIPING,HARBIN,CHINA,45.623,126.250


In [34]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2948 entries, 0 to 9296
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   IATA Code                  2948 non-null   object 
 1   Name                       2948 non-null   object 
 2   City/Town                  2948 non-null   object 
 3   Country                    2948 non-null   object 
 4   Latitude Decimal Degrees   2948 non-null   float64
 5   Longitude Decimal Degrees  2948 non-null   float64
dtypes: float64(2), object(4)
memory usage: 161.2+ KB


In [35]:
# Merge for source airport details with coordinates and location info
merged = df1.merge(
    df2[['IATA Code', 'Name', 'City/Town', 'Country', 
          'Latitude Decimal Degrees', 'Longitude Decimal Degrees']],
    left_on='source airport',
    right_on='IATA Code',
    how='inner'
).rename(columns={
    'Name': 'source airport name',
    'City/Town': 'source city',
    'Country': 'source country',
    'Latitude Decimal Degrees': 'source lat',
    'Longitude Decimal Degrees': 'source lon'
}).drop(columns='IATA Code')

# Merge for destination airport details with coordinates and location info
merged = merged.merge(
    df2[['IATA Code', 'Name', 'City/Town', 'Country',
          'Latitude Decimal Degrees', 'Longitude Decimal Degrees']],
    left_on='destination airport',
    right_on='IATA Code',
    how='inner'
).rename(columns={
    'Name': 'destination airport name',
    'City/Town': 'destination city',
    'Country': 'destination country',
    'Latitude Decimal Degrees': 'destination lat',
    'Longitude Decimal Degrees': 'destination lon'
}).drop(columns='IATA Code')

# Select and order final columns
final_df = merged[[
    'source airport', 'source airport name',
    'source city', 'source country',
    'source lat', 'source lon',
    'destination airport', 'destination airport name',
    'destination city', 'destination country',
    'destination lat', 'destination lon',
    'equipment'
]]
final_df.to_csv('./data/plane_routes_with_coordinates.csv', index=False)
final_df

,source airport,source airport name,source city,source country,source lat,source lon,destination airport,destination airport name,destination city,destination country,destination lat,destination lon,equipment
0,AER,SOCHI,SOCHI,RUSSIA,43.446,39.947,KZN,KAZAN,KAZAN,RUSSIA,55.608,49.277,CR2
1,ASF,ASTRAKHAN,ASTRAKHAN,RUSSIA,46.283,48.006,KZN,KAZAN,KAZAN,RUSSIA,55.608,49.277,CR2
2,ASF,ASTRAKHAN,ASTRAKHAN,RUSSIA,46.283,48.006,MRV,MINERALNYYE VODY,MINERALNYE VODY,RUSSIA,44.225,43.082,CR2
3,CEK,BALANDINO,CHELYABINSK,RUSSIA,55.303,61.507,KZN,KAZAN,KAZAN,RUSSIA,55.608,49.277,CR2
4,KZN,KAZAN,KAZAN,RUSSIA,55.608,49.277,AER,SOCHI,SOCHI,RUSSIA,43.446,39.947,CR2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
50698,TSV,TOWNSVILLE,TOWNSVILLE,AUSTRALIA,-19.253,146.765,ISA,MOUNT ISA,MOUNT ISA,AUSTRALIA,-20.664,139.489,SF3
50699,WGA,WAGGA WAGGA,WAGGA WAGGA,AUSTRALIA,-35.165,147.466,MEL,MELBOURNE INTERNATIONAL,MELBOURNE,AUSTRALIA,-37.673,144.843,SF3
50700,WGA,WAGGA WAGGA,WAGGA WAGGA,AUSTRALIA,-35.165,147.466,SYD,KINGSFORD SMITH INTERNATIONAL AIRPORT,SYDNEY,AUSTRALIA,-33.946,151.177,SF3
50701,FRU,MANAS,BISHKEK,RUSSIA,43.062,74.478,OSS,OSH,OSH,RUSSIA,40.609,72.793,734


In [36]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50703 entries, 0 to 50702
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   source airport            50703 non-null  object 
 1   source airport name       50703 non-null  object 
 2   source city               50703 non-null  object 
 3   source country            50703 non-null  object 
 4   source lat                50703 non-null  float64
 5   source lon                50703 non-null  float64
 6   destination airport       50703 non-null  object 
 7   destination airport name  50703 non-null  object 
 8   destination city          50703 non-null  object 
 9   destination country       50703 non-null  object 
 10  destination lat           50703 non-null  float64
 11  destination lon           50703 non-null  float64
 12  equipment                 50689 non-null  object 
dtypes: float64(4), object(9)
memory usage: 5.0+ MB


### Graficas

In [37]:
# Obtener aeropuertos únicos (combinando origen y destino)
airports_source = final_df[['source airport', 'source airport name', 'source city', 'source country', 'source lat', 'source lon']].rename(columns={
    'source airport': 'airport',
    'source airport name': 'name',
    'source city': 'city',
    'source country': 'country',
    'source lat': 'lat',
    'source lon': 'lon'
})

airports_dest = final_df[['destination airport', 'destination airport name', 'destination city', 'destination country', 'destination lat', 'destination lon']].rename(columns={
    'destination airport': 'airport',
    'destination airport name': 'name',
    'destination city': 'city',
    'destination country': 'country',
    'destination lat': 'lat',
    'destination lon': 'lon'
})

airports = pd.concat([airports_source, airports_dest]
                     ).drop_duplicates(subset=['airport'])

# Preparar coordenadas de las rutas (líneas)
routes_lons = []
routes_lats = []

for _, row in final_df.iterrows():
    routes_lons.extend([row['source lon'], row['destination lon'], None])
    routes_lats.extend([row['source lat'], row['destination lat'], None])

# Calcular operaciones por aeropuerto (origen + destino)
departures = final_df.groupby('source airport').size().reset_index(name='departures')
arrivals = final_df.groupby('destination airport').size().reset_index(name='arrivals')
operations = pd.merge(departures, arrivals, 
                     left_on='source airport', 
                     right_on='destination airport', 
                     how='outer').fillna(0)

operations['total_operations'] = operations['departures'] + operations['arrivals']

# Obtener información geográfica de los aeropuertos
airport_info = final_df[['source airport', 'source airport name', 
                        'source city', 'source country', 
                        'source lat', 'source lon']].drop_duplicates()

operations = pd.merge(operations, airport_info,
                     left_on='source airport',
                     right_on='source airport').drop(columns=['source airport'])

# Crear escala de tamaño y color
max_ops = operations['total_operations'].max()
size_scale = (operations['total_operations'] / max_ops * 30 + 5)  # Tamaño entre 5 y 35

equipment_counts = final_df['equipment'].value_counts().reset_index()
equipment_counts.columns = ['Aeronave', 'Operaciones']
top_20 = equipment_counts.head(20)

In [38]:
# Create figure with professional style
fig = go.Figure()

# Add airports as enhanced markers
fig.add_trace(go.Scattergeo(
    lon = airports['lon'],
    lat = airports['lat'],
    text = airports.apply(lambda x: f"<b>{x['name']}</b><br>{x['city']}, {x['country']}", axis=1),
    mode = 'markers',
    marker = dict(
        size = 6.5,  # Optimized size
        color = '#2A5F7F',  # Professional blue
        opacity = 0.85,
        line = dict(
            width = 1.2,
            color = '#FFFFFF'  # White border for contrast
        )
    ),
    name = 'Airports',
    hoverinfo = 'text',
    hoverlabel = dict(
        bgcolor = 'white',
        font_size = 14,
        font_family = 'Arial'
    )
))

# Professional layout configuration
fig.update_layout(
    title_text = '<b>Global Airport Infrastructure</b><br><sup>Distribution of commercial airports</sup>',
    title_font = dict(
        size = 24,
        family = 'Arial',
        color = '#2A5F7F'
    ),
    showlegend = True,
    legend = dict(
        # title = 'Legend:',
        bgcolor = 'rgba(255, 255, 255, 0.9)',
        bordercolor = '#CCCCCC',
        borderwidth = 1,
        x = 0.01,
        y = 0.98,
        font = dict(
            size = 12,
            color = '#333333'
        )
    ),
    geo = dict(
        scope = 'world',
        projection_type = 'natural earth',
        showland = True,
        landcolor = 'rgb(245, 245, 245)',
        countrycolor = 'rgb(210, 210, 210)',
        showcountries = True,
        showocean = True,
        oceancolor = 'rgb(235, 245, 255)',
        lakecolor = 'rgb(240, 248, 255)'
    ),
    height = 700,
    margin = dict(r=20, l=20, t=100, b=20),
    paper_bgcolor = 'white',
    plot_bgcolor = 'rgba(240, 240, 240, 0.1)',
    hovermode = 'closest'
)

# Add source annotation
fig.add_annotation(
    x = 1,
    y = 0,
    text = "<b>Source:</b> Global air operations data",
    showarrow = False,
    xref = "paper",
    yref = "paper",
    align = "right",
    font = dict(size=10, color='#666666')
)

fig.show()

In [39]:
# Create figure with professional style
fig = go.Figure()

# Add routes as lines with optimization
fig.add_trace(go.Scattergeo(
    lon = routes_lons,
    lat = routes_lats,
    mode = 'lines',
    line = dict(
        width = 0.4,  # Reduce thickness
        color = 'rgba(100, 100, 100, 0.65)',  # Neutral dark gray
    ),
    name = 'Air Routes',
    hoverinfo = 'none'
))

# Add airports as enhanced markers
fig.add_trace(go.Scattergeo(
    lon = airports['lon'],
    lat = airports['lat'],
    text = airports.apply(lambda x: f"<b>{x['name']}</b><br>{x['city']}, {x['country']}", axis=1),
    mode = 'markers',
    marker = dict(
        size = 4.5,
        color = '#E63946',  # Professional red
        opacity = 0.8,
        line = dict(
            width = 0.8,
            color = '#1D3557'  # Dark border for contrast
        )
    ),
    name = 'Airports',
    hoverinfo = 'text'
))

# Add highlighted points for main hubs
top_airports = operations.nlargest(15, 'total_operations')
fig.add_trace(go.Scattergeo(
    lon = top_airports['source lon'],
    lat = top_airports['source lat'],
    text = top_airports.apply(lambda x: f"<b>{x['source airport name']}</b><br>Operations: {x['total_operations']:,}", axis=1),
    mode = 'markers',
    marker = dict(
        size = 12,
        color = '#457B9D',  # Highlight blue
        opacity = 0.9,
        line = dict(width=1, color='white')
    ),
    name = 'Main Hubs',
    hoverinfo = 'text'
))

# Professional layout configuration
fig.update_layout(
    title_text = '<b>Global Airport Network</b><br><sup>Visualization of Air Infrastructure</sup>',
    title_font = dict(size=22, family='Arial', color='#1D3557'),
    showlegend = True,
    legend = dict(
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='#CCCCCC',
        borderwidth=1,
        x=0.01,
        y=0.99
    ),
    geo = dict(
        scope = 'world',
        projection_type = 'natural earth',
        showland = True,
        landcolor = 'rgb(245, 245, 245)',
        countrycolor = 'rgb(210, 210, 210)',
        showcountries = True,
        showocean = True,
        oceancolor = 'rgb(232, 242, 255)'
    ),
    height = 750,
    margin = dict(r=10, l=10, t=90, b=10),
    paper_bgcolor = 'white',
    plot_bgcolor = 'white',
    hoverlabel = dict(
        bgcolor = 'white',
        font_size = 12,
        font_family = 'Arial'
    )
)

# Add density annotation
fig.add_annotation(
    x=0.97,
    y=0.03,
    text = "<b>Route Density:</b><br>Darker lines indicate<br>greater route overlap",
    showarrow = False,
    xref = "paper",
    yref = "paper",
    align = "right",
    font = dict(size=10, color='#666666'),
    bgcolor = "rgba(255, 255, 255, 0.7)",
    bordercolor = "#CCCCCC",
    borderwidth = 1
)

fig.show()

In [40]:
# Create figure
fig = go.Figure()

# Calculate operations per airport

# Marker and layout modifications:
fig.add_trace(go.Scattergeo(
    lon = operations['source lon'],
    lat = operations['source lat'],
    text = operations.apply(lambda x: (
        f"<b>{x['source airport name']}</b><br>"
        f"City: {x['source city']}<br>"
        f"Country: {x['source country']}<br>"
        f"Total operations: {x['total_operations']:,}"
    ), axis=1),
    marker = dict(
        size = size_scale,
        color = operations['total_operations'],
        colorscale = 'sunsetdark',
        line = dict(width=0.5, color='grey'),
        sizemode = 'diameter',
        opacity = 0.8,
        colorbar = dict(
            title = 'Total Operations',
            thickness = 40
        )
    ),
    hoverinfo = 'text',
    name = 'Airports'
))

# Updated layout configuration
fig.update_layout(
    title_text = '<b>Global Air Activity by Airport</b><br><sup>Size and color indicate operational volume</sup>',
    title_font = dict(size=24, family='Arial'),
    geo = dict(
        scope = 'world',
        projection_type = 'natural earth',
        showland = True,
        landcolor = 'rgb(245, 245, 245)',
        countrycolor = 'rgb(200, 200, 200)',
        showcountries = True,
        showocean = True,
        oceancolor = 'rgb(220, 240, 255)'
    ),
    height = 800,
    margin = dict(r=0, l=0, t=80, b=0),
    paper_bgcolor = 'white',
    plot_bgcolor = 'white'
)

# Get top 3 airports
top_3 = operations.nlargest(3, 'total_operations')

# Add top 3 insights
insight_text = []
for i, (_, row) in enumerate(top_3.iterrows()):
    insight_text.append(
        f"<b>#{i+1} {row['source airport name']}</b><br>"
        f"{row['source city']}, {row['source country']}<br>"
        f"<b>{int(row['total_operations']):,} operations</b>"
    )

# Add annotation with top 3
fig.add_annotation(
    x=0.9,
    y=0.01,
    text="<b>Top 3 Busiest Airports</b><br>" + "<br>".join(insight_text),
    showarrow=False,
    xref="paper",
    yref="paper",
    align="left",
    font=dict(size=12, color='#444'),
    bgcolor="rgba(255, 255, 255, 0.8)",
    bordercolor="#666",
    borderwidth=1
)

fig.show()

In [41]:
fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lon = operations['source lon'],
    lat = operations['source lat'],
    text = operations.apply(lambda x: (
        f"<b>{x['source airport name']}</b><br>"
        f"City: {x['source city']}<br>"
        f"Country: {x['source country']}<br>"
        f"Total operations: {x['total_operations']:,}"
    ), axis=1),
    marker = dict(
        size = size_scale,
        color = operations['total_operations'],
        colorscale = 'sunsetdark',
        line = dict(width=0.5, color='grey'),
        sizemode = 'diameter',
        opacity = 0.8,
        colorbar = dict(
            title = 'Total Operations',
            thickness = 40
        )
    ),
    hoverinfo = 'text',
    name = 'Airports'
))

# Zoom to Europe region
fig.update_layout(
    title_text = '<b>Europe Zoom: Air Activity by Airport</b><br><sup>Size and color indicate operational volume</sup>',
    title_font = dict(size=24, family='Arial'),
    geo = dict(
        scope = 'europe',
        projection_type = 'natural earth',
        showland = True,
        landcolor = 'rgb(245, 245, 245)',
        countrycolor = 'rgb(200, 200, 200)',
        showcountries = True,
        showocean = True,
        oceancolor = 'rgb(220, 240, 255)',
        lataxis = dict(range=[34, 72]),
        lonaxis = dict(range=[-25, 45])
    ),
    width = 800,
    height = 600,
    margin = dict(r=0, l=0, t=80, b=0),
    paper_bgcolor = 'white',
    plot_bgcolor = 'white'
)

fig.show()

In [42]:
# Create professional visualization
fig = go.Figure()

fig.add_trace(go.Bar(
    y=top_20['Aeronave'],
    x=top_20['Operaciones'],
    orientation='h',
    marker=dict(
        color='#2c7da0',
        line=dict(color='#1a4d69', width=1.5)
    ),
    texttemplate='%{x:,}',
    textposition='outside',
    hoverinfo='none'
))

# Professional layout configuration
fig.update_layout(
    title=dict(
        text='<b>Top 20 Most Used Aircraft Types</b><br><span style="font-size:14px;color:#5a5a5a">Distribution by number of operations</span>',
        x=0.03,
        y=0.95,
        font=dict(
            family="Arial",
            size=24,
            color="#2c3e50"
        )
    ),
    xaxis=dict(
        title=dict(
            text=None,
            font=dict(
                size=14,
                color='#2c3e50',
                family='Arial'
            )
        ),
        tickfont=dict(size=12, color='#7f8c8d'),
        gridcolor='#dfe6e9',
        showline=True,
        linecolor='#bdc3c7'
    ),
    yaxis=dict(
        autorange="reversed",
        title='Model',
        tickfont=dict(
            size=12,
            color='#2c3e50',
            family='Arial'
        ),
        linecolor='#bdc3c7'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=180, r=50, t=120, b=80),
    height=700,
    width=1000,
    hoverlabel=dict(
        bgcolor="white",
        font_size=14,
        font_family="Arial"
    )
)

# Add professional context elements
fig.add_annotation(
    x=1,
    y=-0.15,
    xref="paper",
    yref="paper",
    text="<span style='font-size:10px;color:#7f8c8d'>Source: Air operations data</span>",
    showarrow=False,
    align="right"
)

# Professional decorative lines
fig.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor='#ecf0f1')
fig.update_yaxes(showgrid=False)

fig.show()

2. Realizar un informe visual de las plantas de energía nuclear, dada la información provista en el archivo `nuclear_power_plants.csv`. El informe deberá mostrar:

    a. La ubicación de las plantas de energía nuclear así como el número de reactores con los que cuenta

    b. La cantidad de plantas y reactores por regiones geográficas

In [43]:
df3 = pd.read_csv('./data/nuclear-power-plants.csv', usecols=['plant', 'region', 'country', 'numreactor', 'latitude', 'longitude'])
df3.loc[df3['country'] == 'KAZAKHSTAN', 'region'] = df3.loc[df3['country'] == 'KAZAKHSTAN', 'region'].fillna('Asia - Middle East and South')
df3.loc[df3['country'] == 'LITHUANIA', 'region'] = df3.loc[df3['country'] == 'LITHUANIA', 'region'].fillna('Europe - Central and Eastern')
df3.loc[df3['country'] == 'NETHERLANDS', 'region'] = df3.loc[df3['country'] == 'NETHERLANDS', 'region'].fillna('Europe - Western')
df3

,region,country,plant,numreactor,latitude,longitude
0,Europe - Western,SWEDEN,AGESTA,1,59.206022,18.082872
1,Europe - Western,SPAIN,ALMARAZ,2,39.808100,-5.696940
2,America - Latin,BRAZIL,ANGRA,3,-23.007857,-44.458098
3,America - Northern,UNITED STATES OF AMERICA,ARKANSAS ONE,2,35.310320,-93.231289
4,Europe - Western,SPAIN,ASCO,2,41.200000,0.566670
...,...,...,...,...,...,...
271,Asia - Far East,CHINA,YANGJIANG,3,21.708333,112.261111
272,America - Northern,UNITED STATES OF AMERICA,YANKEE,1,42.727839,-72.929108
273,Asia - Far East,"KOREA, REPUBLIC OF",YONGGWANG,6,35.411130,126.416190
274,Europe - Central and Eastern,UKRAINE,ZAPOROZHE,6,47.511809,34.585460


In [44]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 276 entries, 0 to 275
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   region      276 non-null    object 
 1   country     276 non-null    object 
 2   plant       276 non-null    object 
 3   numreactor  276 non-null    int64  
 4   latitude    276 non-null    float64
 5   longitude   276 non-null    float64
dtypes: float64(2), int64(1), object(3)
memory usage: 13.1+ KB


In [45]:
country_region_map = {
    'SWEDEN': 'Europe - Western',
    'SPAIN': 'Europe - Western',
    'BRAZIL': 'America - Latin',
    'UNITED STATES OF AMERICA': 'America - Northern',
    'ARGENTINA': 'America - Latin',
    'GERMANY': 'Europe - Western',
    'RUSSIAN FEDERATION': 'Europe - Central and Eastern',
    'BULGARIA': 'Europe - Central and Eastern',
    'FRANCE': 'Europe - Western',
    'UNITED KINGDOM': 'Europe - Western',
    'SWITZERLAND': 'Europe - Western',
    'KAZAKHSTAN': 'Asia - Middle East and South',
    'SLOVAK REPUBLIC': 'Europe - Central and Eastern',
    'NETHERLANDS': 'Europe - Western',
    'CANADA': 'America - Northern',
    'IRAN, ISLAMIC REPUBLIC OF': 'Asia - Middle East and South',
    'ITALY': 'Europe - Western',
    'CHINA': 'Asia - Far East',
    'ROMANIA': 'Europe - Central and Eastern',
    'PAKISTAN': 'Asia - Middle East and South',
    'UKRAINE': 'Europe - Central and Eastern',
    'TAIWAN, CHINA': 'Asia - Far East',
    'BELGIUM': 'Europe - Western',
    'CZECH REPUBLIC': 'Europe - Central and Eastern',
    'JAPAN': 'Asia - Far East',
    'LITHUANIA': 'Europe - Central and Eastern',
    'INDIA': 'Asia - Middle East and South',
    'SOUTH AFRICA': 'Africa',
    'KOREA, REPUBLIC OF': 'Asia - Far East',
    'SLOVENIA': 'Europe - Central and Eastern',
    'MEXICO': 'America - Latin',
    'FINLAND': 'Europe - Western',
    'ARMENIA': 'Asia - Middle East and South',
    'HUNGARY': 'Europe - Central and Eastern'
}

# Obtener top 3 plantas
top_3 = df3.nlargest(3, 'numreactor')

# Crear texto para insights
insight_text = []
for i, (_, row) in enumerate(top_3.iterrows(), 1):
    insight_text.append(
        f"#{i} <b>{row['plant']}</b><br>"
        f"• Country: {row['country']}<br>"
        f"• Region: {row['region']}<br>"
        f"• Num. of Reactors: {row['numreactor']}"
    )

In [46]:
# Configurar valores únicos esperados (1-9 reactores)
num_reactors = list(range(1, 10))
n_bins = len(num_reactors)  # Debería ser 9

# Generar la escala de colores con bins exactos
discrete_colors = px.colors.sample_colorscale(
    'sunsetdark',
    [n/(n_bins-1) for n in range(n_bins)]
)

# Construir la escala de colores con segmentos precisos
discrete_colorscale = []
for i in range(n_bins):
    start = i/n_bins  # Usamos n_bins como divisor
    end = (i+1)/n_bins
    discrete_colorscale.extend([
        [start, discrete_colors[i]],
        [end, discrete_colors[i]]]
    )

# Ajustar los límites de color
cmin = 0.5  # Valor mínimo ajustado
cmax = 9.5   # Valor máximo ajustado

# Modificar el trazado del marcador
marker = dict(
    size = df3['numreactor'] * 3.5,
    color = df3['numreactor'],
    colorscale = discrete_colorscale,
    cmin = cmin,
    cmax = cmax,
    line = dict(width=0.5, color='grey'),
    sizemode = 'diameter',
    opacity = 0.85,
    colorbar = dict(
        title = 'Number of Reactors',
        thickness = 20,
        len = 0.5,
        tickvals = num_reactors,
        ticktext = [str(n) for n in num_reactors],
        dtick = 1,
        tickmode = 'array'
    )
)

# Crear figura
fig = go.Figure()

# Añadir trazado principal
fig.add_trace(go.Scattergeo(
    lon = df3['longitude'],
    lat = df3['latitude'],
    text = df3.apply(lambda x: (
        f"<b>{x['plant']}</b><br>"
        f"Country: {x['country']}<br>"
        f"Region: {x['region']}<br>"
        f"Reactors: {x['numreactor']}"
    ), axis=1),
    marker = marker,
    hoverinfo = 'text',
    name = 'Nuclear Plants'
    )
)

# Layout configuration (keep previous settings)
fig.update_layout(
    title_text = '<b>Global Distribution of Nuclear Plants</b><br><sup>Size and color indicate number of reactors</sup>',
    title_font = dict(size=24, family='Arial'),
    geo = dict(
        scope = 'world',
        projection_type = 'natural earth',
        showland = True,
        landcolor = 'rgb(243, 243, 243)',
        countrycolor = 'rgb(200, 200, 200)',
        showcountries = True,
        showocean = True,
        oceancolor = 'rgb(220, 240, 255)'
    ),
    height = 750,
    margin = dict(r=0, l=0, t=90, b=0),
    paper_bgcolor = 'white',
    plot_bgcolor = 'white'
)

# Add annotation with top 3
fig.add_annotation(
    x=0.02,
    y=0.03,
    text="<b>Top 3 Nuclear Plants by Number of Reactors</b><br>" + "<br>".join(insight_text),
    showarrow=False,
    xref="paper",
    yref="paper",
    align="left",
    font=dict(size=12, color='#444'),
    bgcolor="rgba(255, 255, 255, 0.85)",
    bordercolor="#666",
    borderwidth=1,
    borderpad=4
)

fig.show()

In [47]:
# Filtrar plantas en Europa
df_europe = df3[df3['region'].str.contains('Europe', na=False)]

# Generar top 3 específico para Europa
top_3_europe = df_europe.nlargest(3, 'numreactor')
insight_text_europe = []
for i, (_, row) in enumerate(top_3_europe.iterrows(), 1):
    insight_text_europe.append(
        f"#{i} <b>{row['plant']}</b><br>"
        f"Country: {row['country']}<br>"
        f"Reactors: {row['numreactor']}"
    )

# Determine Europe-specific reactor range
eu_min = df3[df3['region'].str.contains('Europe', na=False)]['numreactor'].min()
eu_max = df3[df3['region'].str.contains('Europe', na=False)]['numreactor'].max()
num_reactors_eu = list(range(eu_min, eu_max + 1))
n_bins_eu = len(num_reactors_eu)

# Generate discrete color scale for Europe range
discrete_colors = px.colors.sample_colorscale(
    'sunsetdark',
    [i/(n_bins_eu-1) for i in range(n_bins_eu)]
)
discrete_colorscale_eu = []
for i in range(n_bins_eu):
    start = i / n_bins_eu
    end = (i + 1) / n_bins_eu
    discrete_colorscale_eu.extend([
        [start, discrete_colors[i]],
        [end, discrete_colors[i]]
    ])

cmin_eu = eu_min - 0.5
cmax_eu = eu_max + 0.5

# Configure marker for Europe zoom with Europe-specific scale
marker = dict(
    size = df3['numreactor'] * 3.5,
    color = df3['numreactor'],
    colorscale = discrete_colorscale_eu,
    cmin = cmin_eu,
    cmax = cmax_eu,
    line = dict(width=0.5, color='grey'),
    sizemode = 'diameter',
    opacity = 0.85,
    colorbar = dict(
        title = 'Number of Reactors',
        thickness = 20,
        len = 0.6,
        tickvals = num_reactors_eu,
        ticktext = [str(n) for n in num_reactors_eu],
        dtick = 1,
        tickmode = 'array'
    )
)

# Create Europe zoom figure using only Europe data
df_europe = df3[df3['region'].str.contains('Europe', na=False)]
fig = go.Figure()
fig.add_trace(
    go.Scattergeo(
        lon = df_europe['longitude'],
        lat = df_europe['latitude'],
        text = df_europe.apply(lambda x: (
            f"<b>{x['plant']}</b><br>"
            f"Country: {x['country']}<br>"
            f"Region: {x['region']}<br>"
            f"Reactors: {x['numreactor']}"
        ), axis=1),
        marker = marker,
        hoverinfo = 'text',
        name = 'European Nuclear Plants'
    )
)

# Update layout for Europe zoom
fig.update_layout(
    title_text = '<b>Europe Zoom: Nuclear Plant Distribution</b><br><sup>Size and color indicate number of reactors</sup>',
    title_font = dict(size=24, family='Arial'),
    geo = dict(
        scope = 'europe',
        projection_type = 'natural earth',
        showland = True,
        landcolor = 'rgb(243, 243, 243)',
        countrycolor = 'rgb(200, 200, 200)',
        showcountries = True,
        showocean = True,
        oceancolor = 'rgb(220, 240, 255)',
        lataxis = dict(range=[34, 72]),
        lonaxis = dict(range=[-25, 45])
    ),
    width = 800,
    height = 600,
    margin = dict(r=10, l=10, t=90, b=10),
    paper_bgcolor = 'white',
    plot_bgcolor = 'white'
)

# Add annotation with top 3 European plants
fig.add_annotation(
    x = 0.01, y = 0.97,
    text = "<b>Top 3 European Nuclear Plants</b><br>" + "<br>".join(insight_text_europe),
    showarrow = False,
    xref = 'paper', yref = 'paper',
    align = 'left',
    font = dict(size=12, color='#444'),
    bgcolor = 'rgba(255,255,255,0.85)',
    bordercolor = '#666', borderwidth = 1, borderpad = 4
)

fig.show()


In [48]:
# Map countries to predefined regions
df3['Region'] = df3['country'].map(country_region_map)

# Aggregate per country: count plants, keep region label
df_country = df3.groupby('country').agg(
    num_plants=('plant', 'count'),
    Region=('Region', 'first')
).reset_index()

# Aggregate per region: sum number of plants
region_agg = df_country.groupby('Region').agg(
    total_plants=('num_plants', 'sum')
).reset_index()
# Map region sums back to country-level DataFrame
df_country['region_plants'] = df_country['Region'].map(
    region_agg.set_index('Region')['total_plants']
)

# Determine dynamic range for region_plants
tot_min = region_agg['total_plants'].min()
tot_max = region_agg['total_plants'].max()
# Generate tick values every 10 (from 0 to next multiple of 10 above tot_max)
top = int(np.ceil(tot_max / 10.0)) * 10
tick_vals = list(range(0, top + 1, 10))

# Create choropleth: continuous color scale
fig = px.choropleth(
    df_country,
    locations='country', locationmode='country names',
    color='region_plants',
    hover_name='country', hover_data={'Region':True, 'num_plants':True, 'region_plants':True},
    color_continuous_scale='sunsetdark',
    range_color=(0, top)
)

# Professional layout
fig.update_layout(
    title_text='<b>Number of Nuclear Plants by Region</b><br><sup>Aggregated per Region</sup>',
    title_font=dict(size=24, family='Arial', color='#1D3557'),
    legend_title_text='Total Plants',
    geo = dict(
        scope = 'world',
        projection_type = 'natural earth',
        showland = True,
        landcolor = 'rgb(243, 243, 243)',
        countrycolor = 'rgb(200, 200, 200)',
        showcountries = True,
        showocean = True,
        oceancolor = 'rgb(220, 240, 255)'
    ),
    coloraxis_colorbar=dict(
        title='Total Plants', thickness=20,
        tickvals=tick_vals, ticktext=[str(v) for v in tick_vals], tickmode='array'
    ),
    height = 750,
    margin = dict(r=0, l=0, t=90, b=0),
    paper_bgcolor = 'white',
    plot_bgcolor = 'white'
)

# Annotation: summary per region
totals = region_agg
summary_text = [f"<b>{row.Region}</b>: {row.total_plants} plants" for row in totals.itertuples()]
fig.add_annotation(
    x=0.02, y=0.02, text='<br>'.join(summary_text),
    showarrow=False, xref='paper', yref='paper', align='left',
    font=dict(size=15, color='#444'),
    bgcolor='rgba(255,255,255,0.85)', bordercolor='#666', borderwidth=1, borderpad=4
)

fig.show()


In [49]:
# Aggregate per country: sum reactors, keep region label
df_country = df3.groupby('country').agg(
    total_reactors=('numreactor', 'sum'),
    Region=('Region', 'first')
).reset_index()

# Aggregate per region: sum total reactors
region_agg = df_country.groupby('Region').agg(
    region_reactors=('total_reactors', 'sum')
).reset_index()
# Map regional reactor sums back to country-level DataFrame
df_country['region_reactors'] = df_country['Region'].map(
    region_agg.set_index('Region')['region_reactors']
)

# Determine continuous colorbar range
tot_min = 0
max_val = region_agg['region_reactors'].max()
top = int(np.ceil(max_val / 10.0)) * 10
tick_vals = list(range(0, top + 1, 20))

# Create choropleth: countries colored by their region's total reactors
fig = px.choropleth(
    df_country,
    locations='country', locationmode='country names',
    color='region_reactors',
    hover_name='country', hover_data={'Region':True, 'total_reactors':True, 'region_reactors':True},
    color_continuous_scale='sunsetdark',
    range_color=(tot_min, top)
)

# Professional layout adaptation
fig.update_layout(
    title_text='<b>Total Nuclear Reactors by Region</b><br><sup>Aggregated per Region</sup>',
    title_font=dict(size=24, family='Arial', color='#1D3557'),
    legend_title_text='Total Reactors',
    geo = dict(
        scope = 'world',
        projection_type = 'natural earth',
        showland = True,
        landcolor = 'rgb(243, 243, 243)',
        countrycolor = 'rgb(200, 200, 200)',
        showcountries = True,
        showocean = True,
        oceancolor = 'rgb(220, 240, 255)'
    ),
    coloraxis_colorbar=dict(
        title='Total Reactors', thickness=20,
        tickvals=tick_vals, ticktext=[str(v) for v in tick_vals], tickmode='array'
    ),
    height = 750,
    margin = dict(r=0, l=0, t=90, b=0),
    paper_bgcolor = 'white',
    plot_bgcolor = 'white'
)

# Annotation: summary per region
totals = region_agg
summary_text = [f"<b>{row.Region}</b>: {row.region_reactors} reactors" for row in totals.itertuples()]
fig.add_annotation(
    x=0.02, y=0.02, text='<br>'.join(summary_text),
    showarrow=False, xref='paper', yref='paper', align='left',
    font=dict(size=15, color='#444'),
    bgcolor='rgba(255,255,255,0.85)', bordercolor='#666', borderwidth=1, borderpad=4
)

fig.show()


3. Realizar una infografía de las muertes de periodistas a nivel global, basada en la informaciónn provista en el archivo `cpj-database.csv`. La infografía deberá mostrar qué países son los mas peligrosos para ejercer el periodismo, así como la relación de la nacionalidad del periodista y el país en el que fue asesinado.

In [50]:
df4 = pd.read_csv('./data/cpj-database.csv', encoding='ISO-8859-1', engine='python', skip_blank_lines=True)
# Drop unnamed empty columns
df4 = df4.loc[:, ~df4.columns.str.contains('^Unnamed')]

# Optional: strip newline and whitespace from string fields
df4 = df4.map(lambda x: x.strip() if isinstance(x, str) else x)

# Crear diccionario de mapeo para nacionalidades especiales
nationality_mapping = {
    'Syrian': 'Syria',
    'Iraqi': 'Iraq',
    'Somali': 'Somalia',
    'Mexican': 'Mexico',
    'Russian': 'Russia',
    'Iranian': 'Iran',
    'Turkish': 'Turkey',
    'Afghan': 'Afghanistan',
    'Palestinian': 'Israel and the Occupied Palestinian Territory',
    'Congolese': 'Democratic Republic of the Congo',
    'Colombian': 'Colombia',
    'American': 'USA',
    'British': 'UK',
    'Philippine': 'Philippines',
    'Indian': 'India',
    'Pakistani': 'Pakistan',
    'Nigerian': 'Nigeria',
    'Honduran': 'Honduras',
    'Angolan': 'Angola',
    'Burmese': 'Myanmar',
    'Japanese': 'Japan',
    'Egyptian': 'Egypt',
    'Israeli': 'Israel and the Occupied Palestinian Territory',
    'Guatemalan': 'Guatemala',
    'Peruvian': 'Peru',
    'Indonesian': 'Indonesia',
    'Bangadesh': 'Bangladesh',  # Corrección de typo
    'The Philippines': 'Philippines',
    'USA': 'United States',
    'UK': 'United Kingdom',
    'Australia': 'Australia',
    'Belgium': 'Belgium',
    'Bosnia-Herzegovina': 'Bosnia-Herzegovina',
    'Burma': 'Myanmar',
    'Chile': 'Chile',
    'Congo': 'Democratic Republic of the Congo',
    'Cuba': 'Cuba',
    'Democratic Republic of Congo': 'Democratic Republic of the Congo',
    'Dominican': 'Dominican Republic',
    'France Congolese': 'Democratic Republic of the Congo',
    'Germany': 'Germany',
    'Israel': 'Israel and the Occupied Palestinian Territory',
    'Israel Occupied': 'Israel and the Occupied Palestinian Territory',
    'Italy': 'Italy',
    'Jordan': 'Jordan',
    'Kenyan': 'Kenya',
    'Malaysia': 'Malaysia',
    'Libyan': 'Libya',
    'Netherlands': 'Netherlands',
    'Norway': 'Norway',
    'Palestinian Territory': 'Israel and the Occupied Palestinian Territory',
    'Qatar': 'Qatar',
    'Slovakia': 'Slovakia',
    'Slovenia': 'Slovenia',
    'Sweden': 'Sweden',
    'Switzerland': 'Switzerland',
    'Syrai': 'Syria',
    'Syrua': 'Syria',
    'The Netherlands': 'Netherlands',
    'Togo': 'Togo',
    'Turkmen': 'Turkmenistan',
    'UAE': 'United Arab Emirates',
    'United Kingdom': 'UK',
    'United States': 'USA',
    'Zaire': 'Zaire',
    'Republic of Congo': 'Democratic Republic of the Congo',
    'UK': 'UK',
    'USA': 'USA'
}

# Función para estandarizar nacionalidades
def clean_nationality(nat):
    if pd.isna(nat):
        return nat
    
    # Dividir nacionalidades múltiples y tomar la primera
    nat = str(nat).split(',')[0].split(';')[0].split('/')[0].strip()
    
    # Aplicar mapeo especial
    if nat in nationality_mapping:
        return nationality_mapping[nat]
    
    # Eliminar sufijos comunes
    for suffix in ['ese', 'ian', 'ish', 'i', 'an']:
        if nat.endswith(suffix):
            base = nat.rsplit(suffix, 1)[0].strip()
            if base in df4['Country Killed'].unique():
                return base
    
    # Dejar valores que ya coinciden con países
    if nat in df4['Country Killed'].unique():
        return nat
    
    return np.nan  # Para nacionalidades no mapeables

# Aplicar limpieza
df4['Nationality_clean'] = df4['Nationality'].apply(clean_nationality)
df4['Nationality'] = df4['Nationality_clean']
df4 = df4.drop('Nationality_clean', axis=1)
df4

,Date,Name,Sex,Country Killed,Organization,Nationality,Medium,Job,Coverage,Freelance,Local/Foreign,Source of Fire,Type of Death,Impunity (for Murder),Taken Captive,Threatened,Tortured
0,1/1/92,Kladoumbaye Maxime,Male,Chad,Agence Tchadienne de Presse,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
1,1/3/92,Ram Singh Biling,Male,India,"Azdi Awaz, Daily Ajit",India,Print,Print Reporter,"Corruption, Human Rights, Politics",Yes,Local,Government Officials,Dangerous Assignment,NaN,Yes,No,No
2,1/15/92,Mustafa Jeha,Male,Lebanon,Al-Amal,Lebanon,Print,Columnist,Politics,No,Local,Political Group,Murder,Yes,No,No,No
3,1/17/92,Paul Jenks,Male,Croatia,European Pressphoto Agency,UK,Print,Photographer,War,Yes,Foreign,Military Officials,Crossfire/Combat-Related,NaN,No,No,No
4,2/9/92,Halit Gungen,Male,Turkey,2000'e Dogru,Turkey,Print,Editor,"Corruption, Human Rights, Politics",No,Local,Government Officials,Murder,Yes,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1874,on an unknown date (death disclosed February 2...,"Fesshaye ""Joshua"" Yohannes",Male,Eritrea,Setit,Eritrea,Print,Editor,Politics,No,Local,Government Officials,Dangerous Assignment,NaN,Yes,No,No
1875,summer 1993,Zikrullo Valiev,NaN,Tajikistan,Khalqi Ovozi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
1876,Unknown,Fadel al-Hadidi,Male,Iraq,Mosul al-Youm; Sada al-Iraq,Iraq,Print,"Columnist, Editor, Print Reporter","Corruption, Human Rights, Politics, War",No,Local,Political Group,Murder,Yes,Yes,No,No
1877,Unknown,Bilal Ahmed Bilal,Male,Syria,Palestine Today TV,Syria,Television,"Broadcast, Producer","Human Rights, Politics, War",No,Local,Government Officials,Dangerous Assignment,NaN,No,No,Yes


In [51]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1879 entries, 0 to 1878
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Date                   1879 non-null   object
 1   Name                   1879 non-null   object
 2   Sex                    1304 non-null   object
 3   Country Killed         1878 non-null   object
 4   Organization           1876 non-null   object
 5   Nationality            1593 non-null   object
 6   Medium                 1303 non-null   object
 7   Job                    1296 non-null   object
 8   Coverage               1265 non-null   object
 9   Freelance              1303 non-null   object
 10  Local/Foreign          1303 non-null   object
 11  Source of Fire         1303 non-null   object
 12  Type of Death          1303 non-null   object
 13  Impunity (for Murder)  875 non-null    object
 14  Taken Captive          1183 non-null   object
 15  Threatened           

In [52]:
# Data Processing
muertes_por_pais = (
    df4['Country Killed']
    .dropna()
    .value_counts()
    .reset_index(name='Deaths')
    .rename(columns={'Country Killed': 'Country'})
)

# Top 5 countries with strategic coordinates for labels
top_paises = muertes_por_pais.head(5).copy()
coordenadas = {
# 'Iraq': {'lat': 25.5, 'lon': 45},
    # 'Syria': {'lat': 40.5, 'lon': 34.6},
    'Iraq': {'lat': 5, 'lon': 65},
    'Syria': {'lat': 40.5, 'lon': -30},
    'Philippines': {'lat': 12.5, 'lon': 122.0},
    'Mexico': {'lat': 24.0, 'lon': -102.0},
    'Pakistan': {'lat': 30.0, 'lon': 70}
}
top_paises['lat'] = top_paises['Country'].map(lambda x: coordenadas[x]['lat'])
top_paises['lon'] = top_paises['Country'].map(lambda x: coordenadas[x]['lon'])

# Create base map
fig = px.choropleth(
    muertes_por_pais,
    locations="Country",
    locationmode="country names",
    color="Deaths",
    color_continuous_scale="reds",
    title="<b>Most Dangerous Countries for Journalists (1992-2023)</b>",
    height=650,
    labels={'Deaths': 'Total Deaths'}
)

# Add data labels for the top 5
fig.add_trace(go.Scattergeo(
    lon = top_paises['lon'],
    lat = top_paises['lat'],
    text = top_paises.apply(lambda x: f"<b>{x['Country']}</b><br>{x['Deaths']} deaths", axis=1),
    mode = 'text',
    textfont = dict(
        family = "Arial Black",
        size = 13,
        color = "black"
    ),
    hoverinfo = 'skip'
))

# Add arrow lines connecting label to actual coords for Iraq and Syria
actual_coords = {
    'Iraq': {'lat': 33.5, 'lon': 43},
    'Syria': {'lat': 35.5, 'lon': 39.6}
}
for country, label_pos in coordenadas.items():
    if country in actual_coords:
        fig.add_trace(go.Scattergeo(
            lon=[label_pos['lon'], actual_coords[country]['lon']],
            lat=[label_pos['lat'] + 5, actual_coords[country]['lat']],
            mode='lines',
            line=dict(width=1.5, color='black'),
            showlegend=False,
            hoverinfo='none'
        ))

# Add explanatory annotations
fig.add_annotation(
    x=0.02,
    y=0.01,
    xref="paper",
    yref="paper",
    text="• Top 5 accounts for 48% of total deaths<br>"
         "• Iraq = 34% more deaths than Philippines<br>"
         "• Pakistan enters top 5 due to border conflicts<br>"
         "• Mexico only American country in top 5",
    showarrow=False,
    font=dict(size=15, color="#333333"),
    align="left",
    bordercolor="#cccccc",
    borderwidth=1,
    borderpad=10,
    bgcolor="white"
)

# Advanced Customization
fig.update_layout(
    geo = dict(
        scope = 'world',
        projection_type = 'natural earth',
        showland = True,
        landcolor = 'rgb(245, 245, 245)',
        countrycolor = 'rgb(210, 210, 210)',
        showcountries = True,
        showocean = True,
        oceancolor = 'rgb(235, 245, 255)',
        lakecolor = 'rgb(240, 248, 255)'
    ),
    coloraxis=dict(
        colorbar=dict(
            title=dict(text="<b>Deaths</b>", font=dict(size=14)),
            thickness=20,
            len=0.5,
            yanchor="top",
            y=0.7
        )
    ),
    height = 700,
    margin = dict(r=20, l=20, t=100, b=20),
    paper_bgcolor = 'white',
    plot_bgcolor = 'rgba(240, 240, 240, 0.1)',
    hovermode = 'closest',
    title=dict(
        x=0.05,
        y=0.95,
        xanchor='left',
        font=dict(size=22, color='#333333')
    )
)

fig.show()

In [53]:
from plotly.colors import hex_to_rgb

relacion_muertes = df4.dropna(subset=['Nationality', 'Country Killed'])[['Nationality', 'Country Killed']]
top_nacionalidades = relacion_muertes['Nationality'].value_counts().nlargest(15).index
top_paises = relacion_muertes['Country Killed'].value_counts().nlargest(15).index

df_filtrado = relacion_muertes[
    relacion_muertes['Nationality'].isin(top_nacionalidades) &
    relacion_muertes['Country Killed'].isin(top_paises)
]
cross_tab = pd.crosstab(df_filtrado['Nationality'], df_filtrado['Country Killed'])
nacionalidades = cross_tab.index.tolist()
paises = cross_tab.columns.tolist()

# Join nodes and create indices
nodes = nacionalidades + paises
node_indices = {node: i for i, node in enumerate(nodes)}

# Build links only if the value exceeds a minimum threshold
sources, targets, values = [], [], []
min_threshold = 2  # show only flows ≥ 2 cases
for nac in nacionalidades:
    for pais in paises:
        val = cross_tab.at[nac, pais]
        if val >= min_threshold:
            sources.append(node_indices[nac])
            targets.append(node_indices[pais])
            values.append(val)

# Assign consistent colors and palette
palette = px.colors.qualitative.Plotly
node_colors = []
for i, node in enumerate(nodes):
    base = palette[0] if node in nacionalidades else palette[1]
    rgb = hex_to_rgb(base)  # converts "#1f77b4" to (31,119,180)
    # Fixed opacity for all
    node_colors.append(f"rgba({rgb[0]},{rgb[1]},{rgb[2]},0.8)")

# ————— Create the figure —————
fig = go.Figure(
    go.Sankey(
        arrangement="freeform",
        orientation="h",
        node=dict(
            pad=30,
            thickness=15,
            line=dict(color="gray", width=0.5),
            label=nodes,
            color=node_colors,
            hovertemplate="%{label}<extra></extra>"
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color="rgba(100,100,100,0.3)",
            hovertemplate="%{source.label} → %{target.label}<br>Cases: %{value}<extra></extra>"
        )
    )
)

# ————— Custom layout —————
fig.update_layout(
    title_text="<b>Top 15 Nationalities and Countries</b><br><span style='font-size:12px;color:#666;'>Nationality → Country of Death</span>",
    title_x=0.5,
    font=dict(family="Roboto, sans-serif", size=12, color="#333"),
    margin=dict(l=20, r=20, t=100, b=20),
    hoverlabel=dict(bgcolor="white", font_size=11, font_family="Arial"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=500,
    width=800,
)

fig.show()


In [61]:
cross_tab

Country Killed,Afghanistan,Algeria,Bangladesh,Brazil,Colombia,Honduras,India,Iraq,Mexico,Pakistan,Philippines,Russia,Somalia,Syria,Turkey
Nationality,,,,,,,,,,,,,,,
Algeria,0,59,0,0,0,0,0,0,0,0,0,0,0,0,0
Bangladesh,0,0,25,0,0,0,0,0,0,0,0,0,0,0,0
Brazil,0,0,0,50,0,0,0,0,0,0,0,0,0,0,0
Colombia,0,0,0,0,57,0,0,0,0,0,0,0,0,0,0
Honduras,0,0,0,0,0,24,0,0,0,0,0,0,0,0,0
India,0,0,0,0,0,0,57,0,0,0,0,0,0,0,0
Iraq,0,0,0,0,0,0,0,218,0,0,0,0,0,3,0
Mexico,0,0,0,0,0,0,0,0,86,0,0,0,0,0,0
Pakistan,0,0,0,0,0,0,0,0,0,81,0,0,0,0,0
